In [7]:
# imports
import os, math, json, random
import torch
import torch.nn as nn
from torch.utils.data import random_split
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, global_mean_pool
import torch.nn.functional as F
from itertools import product
from torch_geometric.nn import MessagePassing, global_add_pool
from torch_geometric.utils import softmax
from torch import scatter_add

In [8]:
# dataset
NODE_FEATURE_DIM = 10
EDGE_FEATURE_DIM = 4

OP_TYPES = ['kn_input_op', 
            'kn_output_op', 
            'kn_add_op', 
            'kn_div_op',
            'kn_exp_op', 
            'kn_matmul_op', 
            'kn_mul_op',
            'kn_pow_op', 
            'kn_reduction_2_op', 
            'kn_sqrt_op']

def one_hot_op_type(op_type):
    out = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    out[OP_TYPES.index(op_type)] = 1.0
    return out

def compute_flops(op_type, input_tensors):
    if op_type == 'kn_matmul_op':
        M = input_tensors[0][-2]
        K = input_tensors[0][-1]
        N = input_tensors[1][-1]
        return 2 * M * N * K
    elif op_type in ['kn_add_op', 
                     'kn_div_op', 
                     'kn_exp_op', 
                     'kn_mul_op', 
                     'kn_pow_op', 
                     'kn_sqrt_op', 
                     'kn_reduction_2_op']:
        num_elements = 1
        for dim in input_tensors[0]:
            num_elements *= dim
        return num_elements
    elif op_type in ['kn_input_op', 'kn_output_op']:
        return 0
    else:
        raise ValueError(f"Unknown op type: {op_type}")

def get_node_features(node):
    # one-hot op type
    h_op_type = one_hot_op_type(node['op_type'])
    
    # in tensor dimensions
    in_tensors = []
    h_in_dims = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(node['input_tensors']):
        in_tensors.append(t['dim'])
        for j, dim in enumerate(t['dim']):
            h_in_dims[j + (i * 4)] = dim
    
    # out tensor dimensions
    out_tensors = []
    h_out_dims = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(node['output_tensors']):
        out_tensors.append(t['dim'])
        for j, dim in enumerate(t['dim']):
            h_out_dims[j + (i * 4)] = dim
    
    # in tensor sizes
    h_in_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(in_tensors):
        h_in_size[i] = math.prod(t)
    
    # out tensor sizes
    h_out_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(out_tensors):
        h_out_size[i] = math.prod(t)

    h_out_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(out_tensors):
        h_out_size[i] = math.prod(t)

    # computation cost in FLOPs
    flops = compute_flops(node['op_type'], in_tensors)
    h_flops = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    h_flops[0] = flops

    return torch.vstack([h_op_type, h_in_dims, h_out_dims, h_in_size, h_out_size, h_flops])

def get_edge_features(tensor):
    h_edge = torch.zeros(EDGE_FEATURE_DIM, dtype=torch.float)
    for i, dim in enumerate(tensor['dim']):
        h_edge[i] = dim
    
    h_edge_size = torch.zeros(EDGE_FEATURE_DIM, dtype=torch.float)
    h_edge_size[0] = math.prod(tensor['dim'])

    return torch.vstack([h_edge, h_edge_size])

class SubgraphDataset(Dataset):
    def __init__(self, root):   
        super().__init__(root)
        json_files = [f for f in os.listdir(root) if f.endswith('.json') and f.startswith('original_')]

        hash_to_time = json.load(open(os.path.join(root, 'performance.json'), 'r'))

        self.subgraph_files = []
        self.execution_times = []
        for json_file in json_files:
            hash = json_file[len('original_'):-len('.json')]
            if hash in hash_to_time:
                self.subgraph_files.append(json_file)
                self.execution_times.append(hash_to_time[hash])
        
        assert len(self.subgraph_files) == len(self.execution_times)

    def len(self):
        return len(self.subgraph_files)
    
    def get(self, idx):
        json_path = os.path.join(self.root, self.subgraph_files[idx])
        json_graph = json.load(open(json_path, 'r'))

        nodes = []
        edge_index = []
        edge_attr = []

        producer_of = {}

        for node_idx, node in enumerate(json_graph):
            nodes.append(get_node_features(node))

            for t in node['output_tensors']:
                producer_of[t['guid']] = node_idx

        for dst_idx, node in enumerate(json_graph):
            for t in node['input_tensors']:
                src_idx = producer_of[t['guid']]
                edge_index.append([src_idx, dst_idx])
                edge_attr.append(get_edge_features(t))
        
        return Data(
            x=torch.stack(nodes, dim=0),
            edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
            edge_attr=torch.stack(edge_attr, dim=0),
            y=torch.tensor([self.execution_times[idx]], dtype=torch.float),
        )

In [14]:
class ANEELayer(MessagePassing):
    """
    Attention-based Node-Edge Encoder (ANEE) layer from DNNPerf (§III-D).
    Update order per round i (paper notation):
      1) h~_u^i = LeakyReLU(W_u h_u^{i-1})
      2) e_l^i = σ(  α(us,ud) ⊙ (W_e e_l^{i-1})  ),  where α(us,ud) = A([h~_us || h~_ud])
      3) α_edge = softmax(W_m e_l^i, dst)  (scalar per edge)
      4) h_u^i = LeakyReLU( Σ_{l'=(u',u)} α_edge(l') * h~_{u'} )
    We implement A as a Linear(2*hidden → hidden) producing a vector gate α to
    modulate the transformed edge feature W_e e; W_m is Linear(hidden→1).
    """
    def __init__(self, hidden: int, edge_in: int, dropout: float = 0.0, share_weights: bool = False):
        super().__init__(aggr="add", node_dim=0)
        self.hidden = hidden
        self.dropout = dropout

        # Node transform W_u
        self.W_u = nn.Linear(hidden, hidden)

        # Edge transforms
        self.W_e = nn.Linear(edge_in, hidden)
        self.A_nodes = nn.Linear(2 * hidden, hidden)  # produces α(us,ud) vector
        self.W_m = nn.Linear(hidden, 1)               # produces attention logit per edge

        self.act_node = nn.LeakyReLU(negative_slope=0.01)
        self.sigmoid = nn.Sigmoid()

        # Optional weight sharing knob kept for parity with paper’s mention
        # (we keep per-layer params by default; set share_weights=True if you
        # want to reuse one instance across message-passing rounds externally).
        self.share_weights = share_weights

    def forward(self, x, edge_index, edge_attr):
        """
        x:         [N, hidden]
        edge_index:[2, E]
        edge_attr: [E, edge_in]
        """
        N = x.size(0)
        src, dst = edge_index

        # 1) Local node update (h~)
        h_tilde = self.act_node(self.W_u(x))  # [N, hidden]

        # Gather pairwise node reps for edges
        h_src = h_tilde[src]  # [E, hidden]
        h_dst = h_tilde[dst]  # [E, hidden]

        # 2) Edge update with node-pair attention
        #    α(us,ud): [E, hidden], W_e e: [E, hidden] → gated and squashed
        alpha_vec = self.A_nodes(torch.cat([h_src, h_dst], dim=-1))        # [E, hidden]
        e_trans   = self.W_e(edge_attr)                                    # [E, hidden]
        e_new     = self.sigmoid(alpha_vec * e_trans)                       # [E, hidden]

        # 3) Edge attention score (scalar), normalized over incoming edges at each dst
        logits = self.W_m(e_new).squeeze(-1)            # [E]
        att    = softmax(logits, dst, num_nodes=N)       # [E]

        # 4) Aggregate edge-weighted neighbor messages
        msg = h_src * att.unsqueeze(-1)                                # [E, hidden]
        # torch.scatter_add needs an `out` tensor and an index with same shape as `msg`
        out = msg.new_zeros((N, self.hidden))                           # [N, hidden]
        dst_index = dst.view(-1, 1).expand(-1, self.hidden)             # [E, hidden]
        out = torch.scatter_add(out, 0, dst_index, msg)                 # [N, hidden]
        
        out = self.act_node(out)
        out = F.dropout(out, p=self.dropout, training=self.training)

        return out


class SubgraphRegressor(nn.Module):
    """
    ANEE-based regressor (replacing TransformerConv with the paper's
    Attention-based Node-Edge Encoder). Keeps the same external API.
      - Flattens node features (6x10 -> 60)
      - Flattens edge features (2x4 -> 8)
      - Stacks ANEE layers
      - Sums node embeddings to a graph embedding (as in the paper)
      - Regresses to a scalar
    """
    def __init__(self, node_in=60, edge_in=8, hidden=128, layers=3, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.node_in = node_in
        self.edge_in = edge_in
        self.hidden  = hidden
        self.layers  = layers

        # Project raw node features to hidden size (unchanged)
        self.x_proj = nn.Sequential(
            nn.Linear(node_in, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
        )

        # Optional projection to align edge features → hidden (the layer also transforms,
        # but this ensures a consistent scale if inputs vary widely)
        self.edge_proj = nn.Linear(edge_in, edge_in)  # identity-capable; keep signature stable

        # --- ANEE stack ---
        self.anee_layers = nn.ModuleList([
            ANEELayer(hidden=hidden, edge_in=edge_in, dropout=dropout)
            for _ in range(layers)
        ])

        # Readout / regressor head
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        # Flatten node features [N,6,10] -> [N,60] if needed
        if x.dim() == 3:
            x = x.view(x.size(0), -1)

        # Flatten edge features [E,2,4] -> [E,8] if needed
        if edge_attr is not None and edge_attr.dim() == 3:
            edge_attr = edge_attr.view(edge_attr.size(0), -1)

        # Node projection to hidden dim
        x = self.x_proj(x)  # [N, hidden]

        # (Optional) small stabilization on edge scale
        if edge_attr is not None:
            edge_attr = self.edge_proj(edge_attr)

        # --- ANEE rounds ---
        for layer in self.anee_layers:
            x = layer(x, edge_index, edge_attr)

        # Graph-level pooling (paper sums node vectors)
        g = global_add_pool(x, batch)  # [num_graphs, hidden]

        # Regression
        out = self.head(g).squeeze(-1)  # [num_graphs]
        return out


In [10]:
# --- helpers ---
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def evaluate(model, loader, device):
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch)
            all_pred.append(pred)
            all_true.append(batch.y.view(-1).to(device))
    if not all_pred:
        return float("nan"), float("nan"), float("nan")
    preds = torch.cat(all_pred)
    trues = torch.cat(all_true)
    mae = (preds - trues).abs().mean().item()
    rmse = torch.sqrt(((preds - trues) ** 2).mean()).item()
    return mae, mae, rmse  # (val_loss proxy, MAE, RMSE)

@torch.no_grad()
def pairwise_accuracy(preds: torch.Tensor, trues: torch.Tensor, exclude_ties: bool = True):
    n = preds.numel()
    if n < 2:
        return float('nan'), 0, 0
    pdiff = preds.view(n, 1) - preds.view(1, n)
    ydiff = trues.view(n, 1) - trues.view(1, n)
    tri_mask = torch.triu(torch.ones(n, n, dtype=torch.bool, device=preds.device), diagonal=1)
    comp_mask = tri_mask & (ydiff != 0) if exclude_ties else tri_mask
    total = int(comp_mask.sum().item())
    if total == 0:
        return float('nan'), 0, 0
    correct = ((torch.sign(pdiff) == torch.sign(ydiff)) & comp_mask).sum().item()
    return correct / total, correct, total

def train_one(model, train_loader, val_loader, device, lr, weight_decay, max_epochs, ckpt_path):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5, verbose=False)

    best_val = float('inf')
    best_state = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss_sum, n_graphs = 0.0, 0
        for batch in train_loader:
            batch = batch.to(device)
            pred = model(batch)
            target = batch.y.view(-1).to(device)
            loss = F.mse_loss(pred, target)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            opt.step()

            train_loss_sum += loss.item() * batch.num_graphs
            n_graphs += batch.num_graphs

        train_mae = train_loss_sum / max(1, n_graphs)

        val_loss, val_mae, val_rmse = evaluate(model, val_loader, device)
        scheduler.step(val_loss)

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f} | Val RMSE: {val_rmse:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = model.state_dict()
            torch.save(best_state, ckpt_path)

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val


def train_eval_grid(
    data_root: str = "./data",
    batch_size: int = 16,
    lr: float = 1e-3,
    weight_decay: float = 1e-5,
    max_epochs: int = 50,
    test_ratio: float = 0.15,   # held-out test
    val_ratio: float = 0.15,    # inner validation (from the remaining train)
    seed: int = 42,
    hidden_grid = (64, 128, 256),
    layers_grid = (2, 3, 4),
    model_prefix: str = "test",
):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- load dataset and make initial train/test split ----
    dataset = SubgraphDataset(data_root)
    n_total = len(dataset)
    assert n_total >= 3, "Need at least 3 samples to create train/val/test splits."

    n_test = max(1, int(round(n_total * test_ratio)))
    n_trainval = n_total - n_test
    gen_main = torch.Generator().manual_seed(seed)
    trainval_ds, test_ds = random_split(dataset, [n_trainval, n_test], generator=gen_main)

    # Pre-build the fixed test loader
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # ---- grid search over hidden × layers using a nested train/val split ----
    best_cfg = None
    best_val_mae = float('inf')
    best_ckpt_tmp = "_grid_best_tmp.pt"

    # indices for deterministic inner split
    n_val = max(1, int(round(len(trainval_ds) * val_ratio)))
    n_train = len(trainval_ds) - n_val
    assert n_train >= 1, "Train portion became empty; reduce val_ratio."

    # Build Subsets for inner train/val once; reuse across configs
    gen_inner = torch.Generator().manual_seed(seed + 1)
    inner_train_subset, inner_val_subset = random_split(trainval_ds, [n_train, n_val], generator=gen_inner)

    inner_train_loader = DataLoader(inner_train_subset, batch_size=batch_size, shuffle=True)
    inner_val_loader   = DataLoader(inner_val_subset,   batch_size=batch_size, shuffle=False)

    for hidden, layers in product(hidden_grid, layers_grid):
        print(f"\n=== Grid config: hidden={hidden}, layers={layers} ===")
        model = SubgraphRegressor(node_in=60, edge_in=8, hidden=hidden, layers=layers, dropout=0.2).to(device)

        # train this config and get best val loss (MAE proxy)
        _ = train_one(model, inner_train_loader, inner_val_loader, device,
                      lr, weight_decay, max_epochs, ckpt_path=best_ckpt_tmp)

        # evaluate on the inner validation split with the best checkpoint
        model.load_state_dict(torch.load(best_ckpt_tmp, map_location=device))
        val_loss, val_mae, val_rmse = evaluate(model, inner_val_loader, device)
        print(f"Config result → Val MAE: {val_mae:.4f} | Val RMSE: {val_rmse:.4f} | Val Loss: {val_loss:.4f}")

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_cfg = {"hidden": hidden, "layers": layers}
            # persist as global best so far
            model_out = f"{model_prefix}_hidden{hidden}_layers{layers}.pt"
            torch.save(model.state_dict(), model_out)
            print(f"  ✔ New best config saved to {model_out}")

    assert best_cfg is not None, "Grid search did not evaluate any configuration."

    print(f"\nBest config found: hidden={best_cfg['hidden']}, layers={best_cfg['layers']} (Val MAE={best_val_mae:.4f})")

    # ---- Test the best configuration on the held-out test set ----
    best_model = SubgraphRegressor(node_in=60, edge_in=8,
                                   hidden=best_cfg["hidden"],
                                   layers=best_cfg["layers"],
                                   dropout=0.2).to(device)
    best_model.load_state_dict(torch.load(model_out, map_location=device))
    test_loss, test_mae, test_rmse = evaluate(best_model, test_loader, device)

    print(f"\n🏁 Test results (held-out): MAE={test_mae:.6f} | RMSE={test_rmse:.6f} | Loss={test_loss:.6f}")

    best_model.eval()
    preds_all, trues_all = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            preds_all.append(best_model(batch))
            trues_all.append(batch.y.view(-1).to(device))
    preds_all = torch.cat(preds_all)
    trues_all = torch.cat(trues_all)

    pair_acc, correct, total = pairwise_accuracy(preds_all, trues_all, exclude_ties=True)
    print(f"🔁 Pairwise comparison accuracy: {pair_acc:.4f} ({correct}/{total} comparable pairs)")

    return {
        "best_config": best_cfg,
        "val_mae": best_val_mae,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "checkpoint": model_out,
    }


In [ ]:
train_eval_grid(data_root='/home/kitao/projects/mirage/dataset/10_16_25_cleaned',
           model_prefix="models/10_27_exec_time",
           hidden_grid=[256, 512, 1024],
           layers_grid=[8, 16, 32])


=== Grid config: hidden=512, layers=8 ===
Epoch 001 | Train MAE: 0.0043 | Val MAE: 0.0286 | Val RMSE: 0.0348
Epoch 010 | Train MAE: 0.0013 | Val MAE: 0.0162 | Val RMSE: 0.0218


KeyboardInterrupt: 